# 03｜Shifted Window：移动窗口与 Attention Mask

上一课学习了普通 Window Attention：把特征图划分成多个固定窗口，每个窗口内部独立计算 Self-Attention。

固定窗口降低了计算量，但也带来一个问题：不同窗口之间无法直接交换信息。

这一课学习 Swin Transformer 最有代表性的设计——Shifted Window，并解释实际实现为什么还需要循环移位和 Attention Mask。

## 1. 固定窗口的问题是什么

假设 Stage 1 的 token 网格为 $56\times56$，窗口大小为 $7\times7$。普通 Window Attention 会得到 $8\times8=64$ 个窗口。

在同一个窗口中，49 个 tokens 可以互相计算注意力；但两个相邻窗口中的 tokens 即使只隔着一条窗口边界，也不能在这一层直接交流。

如果后续每一层都使用完全相同的窗口划分，那么这些边界会一直存在。模型只能学习每个局部窗口内部的关系，却难以组合跨窗口的物体结构。

因此，Swin 需要同时满足两个要求：

1. 每一层仍然只在局部窗口内计算，保留效率优势。
2. 相邻层必须改变窗口分组，让原来分属不同窗口的 tokens 有机会相遇。

## 2. 先用一维序列理解移动窗口

先暂时把二维图片简化成 8 个连续 tokens，窗口长度设为 4。

规则窗口的分组是：

$$
[1,2,3,4]\qquad[5,6,7,8]
$$

token 4 和 token 5 在空间上相邻，却属于不同窗口。

如果下一层把窗口边界移动 2 个位置，新的中间窗口可以包含：

$$
[3,4,5,6]
$$

于是 token 4 和 token 5 进入同一个窗口，可以直接计算 Attention。

二维 Shifted Window 的本质完全相同：同时移动横向和纵向的窗口边界，让上一层不同窗口中的 tokens 在下一层重新组合。

## 3. W-MSA 与 SW-MSA 交替出现

Swin 通常把两个连续的 Transformer Block 配成一组。

第一个 Block 使用规则窗口：

$$
\operatorname{W\text{-}MSA}
$$

第二个 Block 使用移动窗口：

$$
\operatorname{SW\text{-}MSA}
$$

它们内部仍然使用多头 Self-Attention，区别只是窗口如何划分。

对于窗口边长 $M=7$，常用的移动量为：

$$
s=\left\lfloor\frac{M}{2}\right\rfloor=3
$$

也就是沿高度和宽度方向各移动 3 个 token 位置。

## 4. 为什么不直接重新切窗口

从概念上看，只要把窗口网格向右下方移动即可。但直接移动边界后，特征图四周会出现不完整窗口。

完整窗口仍然是 $7\times7$，边缘窗口却可能只有 $3\times7$、$7\times3$ 或 $3\times3$。不同窗口包含的 token 数量不相同，就不容易组成一个规则批次进行并行 Attention。

如果为边缘窗口分别计算，不但实现复杂，而且会降低硬件并行效率。

Swin 因此采用一种等价但更规整的实现方式：先对整张特征图做循环移位，再继续使用与 W-MSA 完全相同的规则窗口划分。

## 5. 循环移位是什么

设输入特征为：

$$
X\in\mathbb{R}^{B\times H\times W\times C}
$$

在 SW-MSA 中，先沿高度和宽度方向各循环移动 $-s$ 个位置。对于 Swin-T Stage 1，$s=3$。

“循环”的含义是：从上边界移出去的部分会回到下边界，从左边界移出去的部分会回到右边界。形状始终保持不变：

$$
B\times56\times56\times96
\xrightarrow{\text{循环移位}}
B\times56\times56\times96
$$

移位后再按照 $7\times7$ 划分，就仍然得到 64 个完整窗口，每个窗口仍然包含 49 个 tokens。

所以循环移位的主要价值是：把不规则的移动窗口重新转化为规则、等大的窗口批次。

## 6. 循环移位带来了什么新问题

循环移位会把特征图一侧的 tokens 绕回另一侧。这样虽然保持了规则窗口，却可能让原本相距很远的边界区域进入同一个窗口。

例如，最右侧的 token 循环后可能出现在最左侧附近。从张量排列上看，它们现在属于同一个窗口；但从原始图片空间看，它们并不是真正的邻居。

如果不加限制，这些不相邻 tokens 就会错误地互相计算 Attention。

因此需要明确区分：

- 移动窗口希望建立的是原窗口边界两侧的真实邻域联系；
- 循环移位额外制造的首尾相接关系只是为了方便计算，不能被当作真实邻域。

Attention Mask 的作用，就是屏蔽第二类错误关系。

## 7. Attention Mask 的核心思想

在循环移位前，可以按照区域来源给 tokens 分配标签。循环移位并划分窗口后，一个窗口中可能同时出现多个来源区域的 tokens。

对于窗口内任意两个 tokens：

- 如果它们来自允许互相通信的同一连续区域，mask 值设为 0；
- 如果它们只是因为循环移位才被拼到一起，mask 值设为一个很大的负数。

单个窗口的 mask 形状为：

$$
M^2\times M^2=49\times49
$$

它与窗口内注意力分数矩阵的最后两个维度一致，因此可以逐位置相加。

## 8. Mask 怎样阻止错误注意力

普通缩放点积注意力分数为：

$$
S=\frac{QK^{\top}}{\sqrt{d_{\mathrm{head}}}}
$$

SW-MSA 会在 Softmax 前加上 mask：

$$
A=\operatorname{softmax}\left(
\frac{QK^{\top}}{\sqrt{d_{\mathrm{head}}}}+\mathcal{M}
\right)
$$

允许通信的位置满足：

$$
\mathcal{M}_{ij}=0
$$

需要屏蔽的位置使用一个很大的负数近似负无穷：

$$
\mathcal{M}_{ij}\approx-\infty
$$

经过 Softmax 后，被屏蔽位置的权重接近 0，因此不会参与对 V 的有效加权汇总。

## 9. Mask 不会屏蔽真正的跨窗口联系

这是最容易误解的地方。

Shifted Window 的目标本来就是让上一层不同窗口中的 tokens 进入同一新窗口。Attention Mask 不会把这些真实相邻关系全部屏蔽掉。

它只屏蔽循环移位造成的虚假首尾连接。

因此，SW-MSA 同时完成两件事：

1. 允许原规则窗口边界附近的 tokens 建立新联系。
2. 阻止图片相反边缘因为循环操作而产生错误联系。

如果把 mask 理解成“所有跨原窗口关系都禁止”，就会失去 Shifted Window 的意义。

## 10. SW-MSA 的完整 shape 路线

以 Swin-T Stage 1 为例，整个过程的形状变化为：

$$
\begin{aligned}
B\times56\times56\times96
&\xrightarrow{\text{循环移位}} B\times56\times56\times96 \\n&\xrightarrow{\text{窗口划分}} (B\cdot64)\times49\times96 \\n&\xrightarrow{\text{QKV 与 Masked Attention}} (B\cdot64)\times49\times96 \\n&\xrightarrow{\text{窗口还原}} B\times56\times56\times96 \\n&\xrightarrow{\text{反向循环移位}} B\times56\times56\times96
\end{aligned}
$$

循环移位后要执行相反方向的移位，把 tokens 放回原来的空间位置。

整个 SW-MSA 前后，$H$、$W$ 和 $C$ 都不改变，所以仍然能够与输入进行残差相加。

## 11. 信息怎样真正跨越窗口

设某个 token 在第一个 W-MSA Block 中只能读取原窗口内部的信息。

进入第二个 SW-MSA Block 后，窗口边界发生变化。这个 token 会与一部分来自相邻原窗口的 tokens 被分到同一新窗口，于是可以读取它们的信息。

经过一组 W-MSA 与 SW-MSA 后，一个 token 的有效感受范围已经超过单个固定窗口。继续堆叠更多 Block，信息还会逐步传播到更远区域。

因此，Swin 并不是在一层内完成全局交互，而是通过多层局部交互逐渐扩大信息传播范围。

这与 CNN 通过堆叠多层卷积逐渐扩大感受野有相似之处，但 Swin 在每个窗口内部使用的是动态 Attention。

## 12. 为什么循环移位是一种高效实现

循环移位并不是 Shifted Window 的学习目标，而是实现移动窗口的一种计算技巧。

它带来三个好处：

1. 移位前后特征图大小不变。
2. 所有窗口仍然是相同的 $7\times7$，可以组成规则批次。
3. W-MSA 与 SW-MSA 可以复用相同的窗口 Attention 计算方式，主要差异只在移位和 mask。

代价是循环边界会制造虚假相邻关系，所以必须配合 Attention Mask。

可以把两者的关系记为：循环移位负责计算规整，Attention Mask 负责空间关系正确。

## 13. 常见误区

### 误区一：Shifted Window 会移动图片内容

模型最终不会改变图片中物体的位置。移位只是 Attention 计算过程中的临时重排，之后还会反向移位恢复原空间位置。

### 误区二：移动以后就变成全局 Attention

每一层仍然只在局部窗口中计算。跨窗口信息是通过相邻 Block 的不同分组逐步传播的。

### 误区三：Mask 会禁止全部跨窗口交流

Mask 只禁止循环移位制造的虚假边界连接，真正由移动窗口建立的相邻区域联系会保留。

### 误区四：循环移位本身完成了信息融合

循环移位只改变排列。真正融合信息的是移位后的 Window Attention。

## 14. 本节小结

这一课需要掌握下面五点：

1. 固定窗口虽然高效，但会阻断不同窗口之间的信息交流。
2. W-MSA 与 SW-MSA 交替使用，让相邻层采用不同的窗口分组。
3. 窗口边长为 7 时，移动量通常为 3。
4. 循环移位让移动后的窗口仍然保持规则大小，Attention Mask 负责屏蔽虚假的首尾连接。
5. SW-MSA 前后 shape 不变，跨窗口信息通过多层局部 Attention 逐步传播。

下一课适合学习相对位置偏置，理解窗口内的 Attention 为什么还需要知道两个 tokens 的相对空间方向和距离。

## 15. 自测问题

1. 固定 Window Attention 为什么会限制跨区域建模？
2. W-MSA 和 SW-MSA 的 Attention 公式是否完全不同？
3. 窗口边长为 7 时，Shifted Window 通常移动多少个 token 位置？
4. 为什么直接移动窗口边界会产生不规则窗口？
5. 循环移位解决了什么计算问题？
6. 循环移位为什么会制造虚假的空间邻居？
7. Attention Mask 中的 0 和大负数分别表示什么？
8. 为什么大负数经过 Softmax 后能屏蔽对应位置？
9. Mask 会不会禁止所有来自不同原窗口的 token 交流？
10. 循环移位以后为什么还要反向移位？
11. SW-MSA 前后的 shape 是否改变？
12. Swin 怎样在不使用单层全局 Attention 的情况下逐渐扩大信息传播范围？

### 自测参考答案

1. 每层都使用相同窗口边界时，不同窗口中的 tokens 无法进入同一次 Attention。
2. 不是。两者都使用窗口内多头 Self-Attention，主要区别是窗口分组方式。
3. 通常移动 $\lfloor7/2\rfloor=3$ 个位置。
4. 移动后的边界不能整齐覆盖特征图四周，会出现尺寸不同的边缘窗口。
5. 它把不规则移动窗口转换为数量和大小都固定的规则窗口，方便并行计算。
6. 一侧移出的 tokens 会绕到另一侧，使原本相距很远的边缘区域暂时进入同一窗口。
7. 0 表示允许通信，大负数表示需要屏蔽。
8. 大负数的指数接近 0，因此对应的 Softmax 权重也接近 0。
9. 不会。它只屏蔽循环操作制造的虚假首尾连接。
10. 为了把更新后的 tokens 放回原来的空间位置。
11. 不改变，输入输出都保持 $B\times H\times W\times C$。
12. 交替使用规则窗口和移动窗口，使 tokens 在相邻层进入不同分组，信息便能逐层跨窗口传播。